# Logistic Regression

This notebook demonstrates a custom logistic regression model trained on a binary classification dataset. The goal is to understand the internal mechanics of the model rather than just using a black-box classifier.

## Understanding the Problem

Binary classification asks a simple question: given an input feature vector, which class is more likely? In this example, we use the Breast Cancer dataset from scikit-learn, where the target is whether a tumor is malignant or benign.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix

from logistic_regression import train_logistic_regression, predict, predict_proba

## Core Concept

Logistic regression models the probability of the positive class using the sigmoid function. The model computes a linear score and then converts it into a value between 0 and 1.

$$
Z = XW + b
$$

$$
at{y} = igma(Z) = rac{1}{1 + e^{-Z}}
$$

A prediction is made by comparing the probability with a threshold such as 0.5.

In [ ]:
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f'X_train shape: {X_train.shape}')
print(f'y_train shape: {y_train.shape}')
print(f'Class balance: {np.bincount(y_train)}')

## Data Preparation

Feature scaling is important because logistic regression uses gradient descent. When features differ greatly in scale, optimization becomes less stable and slower to converge.

In [ ]:
model = train_logistic_regression(
    X_train,
    y_train,
    learning_rate=0.1,
    epochs=1000,
    seed=42,
)

weights = model['weights']
bias = model['bias']
losses = model['losses']

print('Weight shape:', weights.shape)
print('Bias shape:', bias.shape)
print('First 5 losses:', losses[:5])

## Training

The training step updates the weights and bias using gradient descent with the binary cross-entropy loss. The notebook imports the implementation from the Python source file and then runs the learning loop.

In [ ]:
train_probabilities = predict_proba(X_train, weights, bias).ravel()
test_probabilities = predict_proba(X_test, weights, bias).ravel()

train_predictions = predict(X_train, weights, bias).ravel()
test_predictions = predict(X_test, weights, bias).ravel()

train_accuracy = accuracy_score(y_train, train_predictions)
test_accuracy = accuracy_score(y_test, test_predictions)

print(f'Train accuracy: {train_accuracy:.4f}')
print(f'Test accuracy: {test_accuracy:.4f}')
print('Example probabilities:', np.round(test_probabilities[:5], 4))

## Evaluation

After learning, we evaluate the model using accuracy and a confusion matrix. This helps us answer the practical question: how often is the model correct, and what kinds of mistakes does it make?

In [ ]:
cm = confusion_matrix(y_test, test_predictions)
print('Confusion matrix:
', cm)

plt.figure(figsize=(8, 5))
plt.plot(losses, color='royalblue', linewidth=2)
plt.title('Training Loss Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Binary Cross-Entropy')
plt.grid(True, alpha=0.3)
plt.show()

## Technical Analysis

The model learns by minimizing the difference between predicted probabilities and the true labels. Early in training, the loss decreases quickly; later, it settles as the decision boundary becomes more stable. This is a classic example of optimization using gradient descent.

Logistic regression is simple, interpretable, and useful for linearly separable data. However, it cannot model highly non-linear patterns unless additional features or a deeper architecture are used.

## Limitations

This experiment is intentionally compact. The model assumes a linear decision boundary, which is not enough for data with more complex structure. For such cases, a hidden-layer neural network or more expressive representation is required.

## Possible Improvements

- Add regularization to reduce overfitting.
- Try a more informative feature set.
- Compare with a small neural network to see the impact of non-linearity.